In [23]:
import os
from ratelimit import limits, sleep_and_retry
import requests
import json
from urllib3.util import Retry
import sqlite3

In [10]:
conn = sqlite3.connect('league_data.db')
cursor = conn.cursor()

In [24]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS match_queue (
        match_id TEXT PRIMARY KEY
        , status TEXT DEFAULT 'pending')
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS matches (
        match_id TEXT PRIMARY KEY,
        champ_1 TEXT, champ_2 TEXT, champ_3 TEXT, champ_4 TEXT, champ_5 TEXT,
        champ_6 TEXT, champ_7 TEXT, champ_8 TEXT, champ_9 TEXT, champ_10 TEXT)
''')

conn.commit()

In [27]:
EPOCH_TIME_JULY2026 = 1782867600
RANKED_SOLO = 420
ONE_SECOND = 1
TWO_MINUTES = 120
NUM_CHAMPIONS_PER_GAME = 10
NUM_GAMES_PER_PLAYER = 1
CURRENT_PATCH = '16.14.1' # Update this with the current patch version

platforms = ['OC1', 'JP1', 'KR', 'BR1', 'LA1', 'LA2', 'NA1', 'TR1', 'RU', 'EUN1', 'EUW1', 'ME1', 'SG2', 'TW2', 'VN2']
regions = {'OC1':'sea', 'SG2': 'sea', 'TW2': 'sea', 'VN2': 'sea', 'JP1': 'asia', 'KR': 'asia', 'BR1': 'americas', 'LA1': 'americas', 'LA2': 'americas', 'NA1': 'americas', 'TR1': 'europe', 'RU': 'europe', 'EUN1': 'europe', 'EUW1': 'europe', 'ME1': 'europe'}

champion_names_url = 'https://ddragon.leagueoflegends.com/cdn/{version}/data/en_US/champion.json'
master_division_url = 'https://{platform}.api.riotgames.com/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5'
matches_by_player_url = 'https://{region}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?startTime={start_time}&queue={queue}&type=ranked&start=0&count={count}'
match_data_from_matchid = 'https://{region}.api.riotgames.com/lol/match/v5/matches/{matchId}'
api_key = os.getenv("RIOT_API_KEY")

headers = {
    'X-Riot-Token': api_key
}

In [28]:
print(api_key)

RGAPI-232f4127-b394-4aea-bcab-c8829f9f2b96


In [13]:
session = requests.Session()
retries = Retry(total=10,
                backoff_factor=2,
                status_forcelist=[429, 500, 502, 503, 504])
session.mount('https://', requests.adapters.HTTPAdapter(max_retries=retries))

In [14]:
@sleep_and_retry
@limits(calls=95, period=TWO_MINUTES)
@limits(calls=18, period=ONE_SECOND)
def call_api(url, headers=None):
    response = session.get(url, headers=headers)

    if response.status_code >= 400:
        print(f'Status: {response.status_code} Url: {url}')
        return None
    
    return response

In [16]:
for platform in platforms:
    player_data = call_api(master_division_url.format(platform=platform), headers)

    data = player_data.json()['entries']
    player_id = [player['puuid'] for player in data] # collects all the player puuids from the master division

    for puuid in player_id:
        url = matches_by_player_url.format(region=regions[platform],
                                           puuid=puuid,
                                           start_time=EPOCH_TIME_JULY2026,
                                           queue=RANKED_SOLO,
                                           count=NUM_GAMES_PER_PLAYER)
        response = call_api(url, headers)
        
        if not response:
            continue

        for match in response.json():
            cursor.execute('INSERT OR IGNORE INTO match_queue (match) VALUES ?')

        conn.commit()
        print('Added match to queue')

Status: 401 Url: https://OC1.api.riotgames.com/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5


AttributeError: 'NoneType' object has no attribute 'json'

In [ ]:
print(f'Match data collected for {platform}')
    
for match_id in match_data:
    url = match_data_from_matchid.format(region=regions[platform],
                                         matchId=match_id)
        
    response = call_api(url, headers)

    if not response:
        continue
        
    players = response.json()['info']['participants']
    current_champs = [player['championName'] for player in players] # first 5 players are team 1, next 5 are team 2
    champ_data.append(current_champs)

    print(f'{len(match_data) - len(champ_data)} matches left to parse for {platform}')

In [ ]:
all_champion_names = call_api(champion_names_url.format(version=CURRENT_PATCH))

session.close()

In [80]:
with open('champ_names.json', 'w') as f:
    json.dump(list(all_champion_names.json()['data'].keys()), f)

In [63]:
with open('champ_data.json', 'w') as f:
    json.dump(champ_data, f)